In [25]:
import numpy as np
import pandas as pd
import glob, os, vcf, warnings, shutil, subprocess, re, sys, itertools, pysam, vcf
from Bio import Seq, SeqIO
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import scipy, pysam
from matplotlib.colors import ListedColormap
from collections import Counter
from matplotlib.patches import Patch

tbtyper_SNPs_by_lineage = pd.read_csv("~/MtbLongitudinalDiversity/direct_sputum/tbtyper_SNPs_by_lineage.csv")

Coll_2014 = pd.read_csv("/home/sak0914/MtbLongitudinalDiversity/lowAF_variant_calling/references/phylogeny/Coll2014_SNPs_all.csv")
Coll_2014['lineage'] = Coll_2014['lineage'].str.replace('lineage', '')

h37Rv_path = "/n/data1/hms/dbmi/farhat/Sanjana/H37Rv"
h37Rv_seq = SeqIO.read(os.path.join(h37Rv_path, "GCF_000195955.2_ASM19595v2_genomic.gbff"), "genbank")
h37Rv_genes = pd.read_csv(os.path.join(h37Rv_path, "mycobrowser_h37rv_genes_v4.csv"))
h37Rv_regions = pd.read_csv(os.path.join(h37Rv_path, "mycobrowser_h37rv_v4.csv"))

# h37Rv_coords = pd.read_csv(os.path.join(h37Rv_path, "h37Rv_coords_to_gene.csv"))
# h37Rv_coords_dict = dict(zip(h37Rv_coords["pos"].values, h37Rv_coords["region"].values))

In [4]:
bam_file = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_mixed_infection_HP/S0066-01/bam/S0066-01.phased.bam"
# bam_file = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_mixed_infection_HP/S0066-01/bam_2/S0066-01.phased.bam"

bam_in = pysam.AlignmentFile(bam_file, "rb")

ps_tags = []

# Loop through all reads
for read in bam_in:

    if read.has_tag("PS"):
        ps = str(read.get_tag("PS"))
        
        ps_tags.append(ps)

In [5]:
len(ps_tags)

663

In [6]:
np.unique(ps_tags, return_counts=True)

(array(['2181194', '2338194', '4274125', '4358741', '623280', '915482'],
       dtype='<U7'),
 array([136, 113, 118,  88,  87, 121]))

In [56]:
search_pos = tbtyper_SNPs_by_lineage.query("phylotype in ['4.3.2.1', '4.3.2', '4.3', '4']").pos.values
len(search_pos)

262

In [105]:
vcf_file = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_mixed_infection_HP/S0066-01/bam/S0066-01.SNPs.vcf"

vcf_reader = vcf.Reader(filename=vcf_file)

for record in vcf_reader:
    
    if record.POS in Coll_2014.position.values:

        ref_depth = record.samples[0]['AD'][0]
        alt_depths = record.samples[0]['AD'][1:]

        if ref_depth >= 5 and np.sum(alt_depths) >= 5:
            print(record.POS, record.REF, record.ALT)

333569 G [C]
686972 T [C]
1983317 G [A]
2891379 A [C]
3379788 C [G]


In [106]:
Coll_2014.query("position in [333569, 686972, 1983317, 2891379, 3379788]")

,lineage,position,gene_coordinate,allele_change,codon_number,codon_change,amino_acid_change,locus_Id,gene_name,mutation_type,essential,diagnostic_snp
1638,2.2.1.1,2891379,1585,A/C,529,ACC/CCC,T/P,Rv2567,0,nonsyn,nonessential,no
2202,3.1.2.2,333569,2742,G/C,914,GGC/GGG,G/G,Rv0278c,PE_PGRS3,syn,ND,no
4091,4.4.1.2,1983317,1459,G/A,487,CTG/TTG,L/L,Rv1753c,PPE24,syn,ND,no
4196,4.4.2,1983317,1459,G/A,487,CTG/TTG,L/L,Rv1753c,PPE24,syn,ND,no
4888,4.6.2.1,3379788,665,C/G,222,GGG/GCG,G/A,Rv3021c,PPE47,nonsyn,ND,no
5142,4.9,686972,152,T/C,51,TTC/TCC,F/S,Rv0589,mce2A,nonsyn,ND,no


In [97]:
record.samples[0]['AD']

[1, 126, 1]

In [99]:
record.POS, record.REF, record.ALT

(14861, 'G', [T, C])

In [89]:
bam_file = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_mixed_infection_HP/S0066-01/bam/S0066-01.phased.bam"

bam_in = pysam.AlignmentFile(bam_file, "rb")

for pileupcolumn in bam_in.pileup(stepper="all", truncate=True):
    
    pos = pileupcolumn.reference_pos  # 0-based genomic position
    
    if pos in tbtyper_SNPs_by_lineage.pos.values:
        
        # ref = h37Rv_seq[pos]
        base_counts = []

        for pileupread in pileupcolumn.pileups:
            if pileupread.is_del or pileupread.is_refskip:
                continue  # skip deletions and skipped regions

            # get the base supported by each read if the base quality is ≥ 30
            if pileupread.alignment.query_qualities[pileupread.query_position] >= 30:
                base_counts.append(pileupread.alignment.query_sequence[pileupread.query_position])

        bases, counts = np.unique(base_counts, return_counts=True)
        base_counts_dict = dict(zip(bases, counts))

        # check that there are at least 2 alleles supported by more than 1 read
        if sum(np.array(list(base_counts_dict.values())) > 1) > 1:
            print(pos + 1, ref, base_counts_dict)

2637962 C {'A': 11, 'C': 75}
2830526 G {'A': 3, 'G': 80}
3367766 C {'A': 2, 'C': 96}


In [ ]:
bcftools mpileup S0066-01.bam -q 1 -Q 20 -d 200 -f $ref_genome -a FMT/AD,FMT/DP,FMT/ADF,FMT/ADR,FMT/SCR -Ou --threads 1 | bcftools call --ploidy 1 -A -m --prior 1e-2 -Ov --threads 1 > S0066-01.vcf




In [66]:
199471 in tbtyper_SNPs_by_lineage.pos.values

False

In [67]:
199471 in search_pos

False

In [65]:
tbtyper_SNPs_by_lineage.query("phylotype in ['4.3.2.1', '4.3.2', '4.3', '4']")

,chrom,pos,ref,alt,genotype,phylotype,reference
13,AL123456.3,5520,C,T,1,4.3.2.1,Napier et al. 2020
20,AL123456.3,7222,C,T,1,4.3.2.1,Napier et al. 2020
60,AL123456.3,14251,G,A,1,4.3,Napier et al. 2020
165,AL123456.3,41257,A,G,1,4.3.2,Napier et al. 2020
187,AL123456.3,47699,G,A,1,4.3.2,Napier et al. 2020
...,...,...,...,...,...,...,...
10680,AL123456.3,4349688,C,T,1,4.3.2,Napier et al. 2020
10682,AL123456.3,4350446,G,C,1,4.3.2,Napier et al. 2020
10816,AL123456.3,4388153,G,A,1,4.3.2.1,Napier et al. 2020
10822,AL123456.3,4389202,C,T,1,4.3.2,Napier et al. 2020


In [49]:
pileupread.alignment.query_qualities[pileupread.query_position]

93

In [48]:
pos, pileupread.query_position

(2283218, 232)

In [35]:
pos

5520

In [32]:
Coll_2014.query("position==2181194")

,lineage,position,gene_coordinate,allele_change,codon_number,codon_change,amino_acid_change,locus_Id,gene_name,mutation_type,essential,diagnostic_snp


In [43]:
bases

array(['G'], dtype='<U1')

{'G': 148}

In [4]:
        bcftools mpileup {input.merged_bam_file} -q 1 -Q 10 -d 1000 -f {params.ref_genome} -a FMT/AD -Ou --threads {threads} \
          | bcftools call --ploidy 1 -A -m --prior 1e-2 -C alleles -T {params.tbtypeR_targets} -Ou --threads {threads} \
          | bcftools annotate -x INFO,^FORMAT/GT,^FORMAT/AD -Ov -o {output.tbtypeR_VCF_file_init} --threads {threads}
          
        # rename Chromosome to NC_000962.3 for compatibility
        sed 's/Chromosome/NC_000962.3/g' {output.tbtypeR_VCF_file_init} | gzip -c > {output.tbtypeR_VCF_file}

Haplotype 1: 46956 reads
Haplotype 2: 38348 reads
9942 reads without a haplotype


In [ ]:
bcftools mpileup {input.merged_bam_file} -q 1 -Q 10 -d 1000 -f {params.ref_genome} -a FMT/AD -Ou --threads {threads}

In [ ]:
#CHROM  POS     ID      REF     ALT     QUAL    FILTER  INFO    FORMAT  PacBio.alnH37Rv.bam

In [67]:
vcf_file = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_haplotype_phasing/MFS-313/phased_variants/phased.vcf"

df_variants = pd.DataFrame(columns = ['CHROM', 'POS', 'REF', 'ALT', 'QUAL', 'DP', 'REF_DEPTH', 'ALT_DEPTH', 'PHASE_SET'])
i = 0

vcf_reader = vcf.Reader(filename=vcf_file)

for record in vcf_reader:
    
    assert len(record.samples) == 1
    
    try:
        ps = record.samples[0]['PS']
    except:
        ps = None

    assert len(record.ALT) == 1
    
    df_variants.loc[i, :] = [record.CHROM,
                             record.POS,
                             record.REF,
                             record.ALT[0],
                             record.QUAL,
                             record.samples[0]['DP'],
                             ref_depth,
                             alt_depth,
                             ps
                            ]
    
    i += 1
    
df_variants['AF'] = df_variants['ALT_DEPTH'] / df_variants['DP']

In [77]:
# usually fixed in the sample, so they're found in all lineages in the sample
variants_no_set = len(df_variants.loc[pd.isnull(df_variants['PHASE_SET'])])
print(f"{variants_no_set} are not assigned to a phase set")

662 are not assigned to a phase set


In [46]:
variants_pos = phase_sets_variants[1591]

tbtyper_SNPs_by_lineage.query("pos in @variants_pos").phylotype.value_counts()

phylotype
4           27
4.3.2       21
2           20
4.3.2.1     18
4.3         12
2.2.M1.1     4
2.2          4
6.2.2        1
2.2.1        1
2.2.M4.2     1
Name: count, dtype: int64

In [56]:
Coll_2014.query("position in @variants_pos & diagnostic_snp=='yes'")#.lineage.value_counts()

,lineage,position,gene_coordinate,allele_change,codon_number,codon_change,amino_acid_change,locus_Id,gene_name,mutation_type,essential,diagnostic_snp
1126,2,497491,810,G/A,270,GAC/GAT,D/D,Rv0411c,glnH,syn,essential,yes
1135,2,811753,12,C/T,4,CAC/CAT,H/H,Rv0715,rplX,syn,essential,yes
1595,2.2.1,797736,804,C/T,268,CTC/CTT,L/L,Rv0697,0,syn,essential,yes
2373,4,325505,939,T/C,313,GTT/GTC,V/V,Rv0270,fadD2,syn,nonessential,yes
2383,4,599868,2670,A/G,890,CGA/CGG,R/R,Rv0507,mmpL2,syn,nonessential,yes
2391,4,931123,171,T/C,57,TAT/TAC,Y/Y,Rv0835,lpqQ,syn,nonessential,yes
3311,4.3,764995,1626,C/G,542,GCC/GCG,A/A,Rv0668,rpoC,syn,essential,yes
3581,4.3.2.1,7222,1983,C/T,661,AGC/AGT,S/S,Rv0005,gyrB,syn,essential,yes
3595,4.3.2.1,784581,2097,G/C,699,ACG/ACC,T/T,Rv0684,fusA1,syn,essential,yes
3597,4.3.2.1,1055672,1014,C/A,338,CTG/CTT,L/L,Rv0946c,pgi,syn,essential,yes


In [84]:
phase_set_SNPs = df_variants.query("PHASE_SET==1591 & REF.str.len() == ALT.str.len()").sort_values('AF').POS.values

In [87]:
Coll_2014.query("position in @phase_set_SNPs & diagnostic_snp=='yes'")#.lineage.value_counts()")

,lineage,position,gene_coordinate,allele_change,codon_number,codon_change,amino_acid_change,locus_Id,gene_name,mutation_type,essential,diagnostic_snp
1126,2,497491,810,G/A,270,GAC/GAT,D/D,Rv0411c,glnH,syn,essential,yes
1135,2,811753,12,C/T,4,CAC/CAT,H/H,Rv0715,rplX,syn,essential,yes
1595,2.2.1,797736,804,C/T,268,CTC/CTT,L/L,Rv0697,0,syn,essential,yes
2373,4,325505,939,T/C,313,GTT/GTC,V/V,Rv0270,fadD2,syn,nonessential,yes
2383,4,599868,2670,A/G,890,CGA/CGG,R/R,Rv0507,mmpL2,syn,nonessential,yes
2391,4,931123,171,T/C,57,TAT/TAC,Y/Y,Rv0835,lpqQ,syn,nonessential,yes
3311,4.3,764995,1626,C/G,542,GCC/GCG,A/A,Rv0668,rpoC,syn,essential,yes
3581,4.3.2.1,7222,1983,C/T,661,AGC/AGT,S/S,Rv0005,gyrB,syn,essential,yes
3595,4.3.2.1,784581,2097,G/C,699,ACG/ACC,T/T,Rv0684,fusA1,syn,essential,yes
3597,4.3.2.1,1055672,1014,C/A,338,CTG/CTT,L/L,Rv0946c,pgi,syn,essential,yes


In [88]:
df_variants.query("POS==797736")

,CHROM,POS,REF,ALT,QUAL,DP,REF_DEPTH,ALT_DEPTH,PHASE_SET,AF
1086,Chromosome,797736,C,T,60,253,127,62,1591,0.245059
